# 26 — Single-Concentration Data as Pseudo-pEC50 Labels

Converts single-concentration log2FC activity readouts to estimated pEC50
values and combines with CRC training data. Tests whether the 10× larger
single-conc dataset (21K compounds) improves scaffold CV RAE despite
crude label quality.

Mapping: log2FC → pseudo-pEC50 via logistic regression fit on the
~300 compounds present in both datasets. Down-weights pseudo-labels.

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import HuberRegressor

from pxr.data import load_train, load_test, load_single_conc
from pxr.chem import bemis_murcko, standardize_smiles  # standardize_smiles returns str, not Mol
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.featurize import combined, impute
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
print('Setup complete.')

Setup complete.


In [2]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()
sc    = load_single_conc()

print(f'CRC train: {len(train):,}  |  Single-conc: {len(sc):,}  |  Test: {len(te):,}')
print(f'Single-conc columns: {sc.columns.tolist()}')
print(sc[['log2_fc_estimate', 'fdr_bh']].describe().round(3))

CRC train: 4,139  |  Single-conc: 21,003  |  Test: 513
Single-conc columns: ['name', 'batch', 'smiles', 'plate_id', 'compound_class', 'concentration_M', 'log2_fc_estimate', 'log2_fc_stderr', 't_statistic', 'p_value', 'fdr_bh', 'neg_log10_fdr', 'median_log2_fc', 'n_replicates', 'cohens_d', 'experiment_name', 'ocnt_id', 'split', 'source']
       log2_fc_estimate     fdr_bh
count         21003.000  21003.000
mean              0.505      0.215
std               0.536      0.311
min              -1.226      0.000
25%               0.111      0.000
50%               0.349      0.016
75%               0.805      0.373
max               4.239      1.000


In [3]:
# ── 3. Calibrate log2FC → pseudo-pEC50 on overlapping compounds ──────────────
# standardize_smiles returns canonical SMILES string (not Mol) — needed for merge + combined()
train_std = train.copy()
train_std['std_smiles'] = train_std['smiles'].map(standardize_smiles)
sc_std = sc.copy()
sc_std['std_smiles'] = sc_std['smiles'].map(standardize_smiles)

overlap = train_std.merge(sc_std[['std_smiles', 'log2_fc_estimate', 'fdr_bh']], on='std_smiles', how='inner')
print(f'Overlap (train ∩ single-conc): {len(overlap):,} compounds')

if len(overlap) >= 20:
    cal = HuberRegressor(epsilon=1.5, max_iter=1000)
    cal.fit(overlap[['log2_fc_estimate']], overlap['pec50'])
    print(f'Calibration: pEC50 = {cal.coef_[0]:.3f} * log2FC + {cal.intercept_:.3f}')
    print(f'Pearson r on overlap: {np.corrcoef(overlap["log2_fc_estimate"], overlap["pec50"])[0,1]:.3f}')
else:
    print('Too few overlapping compounds, using simple threshold mapping')
    cal = None

Overlap (train ∩ single-conc): 5,722 compounds
Calibration: pEC50 = 0.496 * log2FC + 4.357
Pearson r on overlap: 0.524


In [4]:
# ── 4. Build pseudo-label dataset from single-conc ────────────────────────────
# Only use high-confidence single-conc readouts (FDR < 0.1 or |log2FC| > 1)
sc_conf = sc_std[
    (sc_std['fdr_bh'] < 0.1) | (sc_std['log2_fc_estimate'].abs() > 1.0)
].copy()
print(f'High-confidence single-conc: {len(sc_conf):,}')

# Remove overlap with CRC train (avoid double-counting)
train_std_set = set(train_std['std_smiles'].dropna())
sc_conf = sc_conf[~sc_conf['std_smiles'].isin(train_std_set)].copy()
print(f'After removing CRC train overlap: {len(sc_conf):,}')

# Map to pseudo-pEC50
if cal is not None:
    sc_conf['pseudo_pec50'] = cal.predict(sc_conf[['log2_fc_estimate']])
else:
    mean_pec50 = train['pec50'].mean()
    std_pec50  = train['pec50'].std()
    # Rough linear scaling: log2FC range roughly maps to pEC50 range
    sc_log2_mean = sc_conf['log2_fc_estimate'].mean()
    sc_log2_std  = sc_conf['log2_fc_estimate'].std()
    sc_conf['pseudo_pec50'] = mean_pec50 + (sc_conf['log2_fc_estimate'] - sc_log2_mean) / (sc_log2_std + 1e-8) * std_pec50

# Clip to training range ± 1
sc_conf['pseudo_pec50'] = sc_conf['pseudo_pec50'].clip(
    train['pec50'].min() - 0.5, train['pec50'].max() + 0.5
)

print(f'Pseudo-pEC50: mean={sc_conf["pseudo_pec50"].mean():.3f}  std={sc_conf["pseudo_pec50"].std():.3f}')
print(sc_conf['pseudo_pec50'].describe().round(3))

High-confidence single-conc: 12,777
After removing CRC train overlap: 7,309
Pseudo-pEC50: mean=4.615  std=0.135
count    7309.000
mean        4.615
std         0.135
min         3.749
25%         4.522
50%         4.591
75%         4.699
max         5.521
Name: pseudo_pec50, dtype: float64


In [5]:
# ── 5. Featurize all datasets ──────────────────────────────────────────────────
smiles_tr  = train['smiles'].tolist()
smiles_sc  = sc_conf['std_smiles'].tolist()
smiles_te  = te['smiles'].tolist()
y_tr       = train['pec50'].values
y_sc       = sc_conf['pseudo_pec50'].values

print('Featurizing CRC train...')
X_tr = impute(combined(smiles_tr))
print('Featurizing single-conc...')
X_sc = impute(combined(smiles_sc))
print('Featurizing test...')
X_te = impute(combined(smiles_te))

print(f'Train: {X_tr.shape}  |  SC: {X_sc.shape}  |  Test: {X_te.shape}')

Featurizing CRC train...


Featurizing single-conc...


Featurizing test...


Train: (4139, 2265)  |  SC: (7309, 2265)  |  Test: (513, 2265)


In [6]:
# ── 6. Scaffold 5-fold CV on CRC train (pseudo-labels as extra training) ──────
# Sample weights: CRC=1.0, pseudo-label=0.25 (lower quality)
SC_WEIGHT = 0.25

LGBM_PARAMS = dict(
    n_estimators=1200, num_leaves=64, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.2,
    min_child_samples=10, n_jobs=4, verbose=-1
)

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    # Combine CRC fold train with all pseudo-labels
    X_fold = np.vstack([X_tr[tr_idx], X_sc])
    y_fold = np.concatenate([y_tr[tr_idx], y_sc])
    w_fold = np.concatenate([
        np.ones(len(tr_idx)),
        np.full(len(y_sc), SC_WEIGHT)
    ])
    
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_fold, y_fold, sample_weight=w_fold)
    oof[va_idx] = m.predict(X_tr[va_idx])
    fold_rae = rae_fn(y_tr[va_idx], oof[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    fold_metrics.append(met)
    print(f'  Fold {fold_i+1}: RAE={fold_rae:.4f}  Spearman={met["Spearman"]:.4f}')

oof_rae = rae_fn(y_tr, oof)
cv_df   = pd.DataFrame(fold_metrics)
print(f'\nOOF RAE (global): {oof_rae:.4f}')
print(f'Mean fold RAE: {cv_df["RAE"].mean():.4f} +/- {cv_df["RAE"].std():.4f}')
print(f'\n== Comparison ==')
print(f'  LGBM_base (no pseudo-labels): ~0.575')
print(f'  LGBM_tuned (no pseudo-labels): 0.5394')
print(f'  Single-conc augmented (this):  {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_singleconc.npy', oof)

  Fold 1: RAE=0.5749  Spearman=0.7812


  Fold 2: RAE=0.6114  Spearman=0.7150


  Fold 3: RAE=0.6153  Spearman=0.7061


  Fold 4: RAE=0.5973  Spearman=0.7196


  Fold 5: RAE=0.6219  Spearman=0.7134

OOF RAE (global): 0.6003
Mean fold RAE: 0.6042 +/- 0.0187

== Comparison ==
  LGBM_base (no pseudo-labels): ~0.575
  LGBM_tuned (no pseudo-labels): 0.5394
  Single-conc augmented (this):  0.6003


In [7]:
# ── 7. Full retrain + test predictions ────────────────────────────────────────
X_full = np.vstack([X_tr, X_sc])
y_full = np.concatenate([y_tr, y_sc])
w_full = np.concatenate([np.ones(len(y_tr)), np.full(len(y_sc), SC_WEIGHT)])

final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_full, y_full, sample_weight=w_full)
te_preds = final_m.predict(X_te)
te_preds = np.clip(te_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_singleconc.npy', te_preds)

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '26_singleconc_lgbm.csv'
sub.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'OOF RAE: {oof_rae:.4f}')
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\26_singleconc_lgbm.csv
OOF RAE: 0.6003
count    513.000
mean       4.858
std        0.503
min        3.048
25%        4.620
50%        4.908
75%        5.196
max        6.011
Name: pEC50, dtype: float64
